In [1]:
# prompt: any suggestions to improve the performance

import pandas as pd
import numpy as np
import plotly.express as px
import torch
import plotly.io as pio
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split as tts
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset



In [2]:
# Set the renderer for displaying plots in Colab
pio.renderers.default = 'colab'

house = pd.read_csv('/content/Chennai house data.csv')
house.head()

,price,area,status,bhk,bathroom,age,location,builder
0,37.49,872,Ready to move,2,NaN,1.0,Sembakkam,MP Developers
1,93.54,1346,Under Construction,3,2.0,NaN,Selaiyur,DAC Promoters
2,151.00,2225,Under Construction,3,NaN,0.0,Mogappair,Casagrand Builder Private Limited
3,49.00,1028,Ready to move,2,2.0,3.0,Ambattur,Dugar Housing Builders
4,42.28,588,Under Construction,2,1.0,0.0,Pallavaram,Radiance Realty Developers India Ltd


In [3]:
house.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2620 entries, 0 to 2619
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   price     2620 non-null   float64
 1   area      2620 non-null   int64  
 2   status    2620 non-null   object 
 3   bhk       2620 non-null   int64  
 4   bathroom  1403 non-null   float64
 5   age       1729 non-null   float64
 6   location  2620 non-null   object 
 7   builder   2620 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 163.9+ KB


In [4]:
# --- Data Preprocessing ---

# Handle missing values
# Consider more advanced imputation if needed
house['bathroom'] = house['bathroom'].bfill()
house['age'] = house['age'].ffill()
house.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2620 entries, 0 to 2619
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   price     2620 non-null   float64
 1   area      2620 non-null   int64  
 2   status    2620 non-null   object 
 3   bhk       2620 non-null   int64  
 4   bathroom  2620 non-null   float64
 5   age       2620 non-null   float64
 6   location  2620 non-null   object 
 7   builder   2620 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 163.9+ KB


In [5]:
numerical_cols = ['price', 'area', 'bhk', 'bathroom', 'age']
for col in numerical_cols:
    fig = px.box(house, y=col, title=f'Boxplot of {col}') # Use px.box
    fig.show() # Explicitly show the figure

In [6]:
# Outlier Handling (using capping as before)
numerical_cols = ['price', 'area', 'bhk', 'bathroom', 'age']
for col in numerical_cols:
    Q1 = house[col].quantile(0.25)
    Q3 = house[col].quantile(0.75)
    IQR = Q3 - Q1
    lbound = Q1 - 1.5 * IQR
    ubound = Q3 + 1.5 * IQR
    house[col] = np.where(house[col] > ubound, ubound,
                            np.where(house[col] < lbound, lbound, house[col]))

In [7]:
for col in numerical_cols:
    fig = px.box(house, y=col, title=f'Boxplot of {col}') # Use px.box
    fig.show() # Explicitly show the figure

In [8]:
# Convert categorical features
# Consider One-Hot Encoding for 'location' and 'builder'
house["status"] = house["status"].map({'Under Construction':0,'Ready to move':1})

le = LabelEncoder() # Consider OneHotEncoder instead
house["location"] = le.fit_transform(house["location"])
house["builder"] = le.fit_transform(house["builder"])

# Separate features and target
Y = house['price']
X = house.drop(columns = ['price'])

# Scaling features
# Using StandardScaler after train/validation/test split is generally better to prevent data leakage
# For demonstration, applying before split here, but recommend applying after split
scaler = StandardScaler() # Or MinMaxScaler
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(data = X_scaled, columns = X.columns)

In [9]:
# --- Data Splitting ---
# Split into training, validation, and testing sets
x_train, x_temp, y_train, y_temp = tts(X_scaled, Y, test_size = 0.2, random_state = 100) # Use 20% for temp
x_val, x_test, y_val, y_test = tts(x_temp, y_temp, test_size = 0.5, random_state = 100) # Split temp into val and test (10% each)

print(f'Training data shape: {x_train.shape}, {y_train.shape}')
print(f'Validation data shape: {x_val.shape}, {y_val.shape}')
print(f'Testing data shape: {x_test.shape}, {y_test.shape}')

# Convert data to PyTorch tensors
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
x_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# Create DataLoader for batch training
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

batch_size = 32 # Experiment with different batch sizes
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

Training data shape: (2096, 7), (2096,)
Validation data shape: (262, 7), (262,)
Testing data shape: (262, 7), (262,)


In [10]:
# --- Define the neural network model with Dropout ---
class Net(nn.Module):
    def __init__(self, input_features):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_features, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2) # Add dropout
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2) # Add dropout
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Instantiate the model
input_features = x_train.shape[1]
model = Net(input_features)

# Define loss function and optimizer (add weight_decay for L2 regularization)
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # Add weight decay

# --- Training loop with validation ---
epochs = 500
train_loss_history = []
val_loss_history = []

for epoch in range(epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for inputs, targets in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_loss_history.append(train_loss)

    # Evaluate on validation set
    model.eval() # Set the model to evaluation mode
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

    val_loss = val_loss / len(val_loader)
    val_loss_history.append(val_loss)

    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

# --- Evaluate the model on the test set ---
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_loader:
        test_outputs = model(inputs)
        loss = criterion(test_outputs, targets)
        test_loss += loss.item()

test_loss = test_loss / len(test_loader)
print(f'Final Test Loss: {test_loss:.4f}')



Epoch [50/500], Train Loss: 556.1936, Val Loss: 429.0876
Epoch [100/500], Train Loss: 515.5076, Val Loss: 401.8507
Epoch [150/500], Train Loss: 511.3320, Val Loss: 379.6865
Epoch [200/500], Train Loss: 482.2233, Val Loss: 359.2870
Epoch [250/500], Train Loss: 495.5910, Val Loss: 355.4947
Epoch [300/500], Train Loss: 500.4501, Val Loss: 346.1397
Epoch [350/500], Train Loss: 469.8746, Val Loss: 337.7529
Epoch [400/500], Train Loss: 462.5638, Val Loss: 338.4789
Epoch [450/500], Train Loss: 444.4982, Val Loss: 314.0208
Epoch [500/500], Train Loss: 460.3396, Val Loss: 320.2426
Final Test Loss: 319.2559


In [11]:
# --- Plotting Results ---

# Plot the training and validation loss
fig = px.line(y=[train_loss_history, val_loss_history], title='Training and Validation Loss over Epochs',
              labels={'value':'Loss', 'index':'Epoch', 'variable':'Loss Type'})
fig.data[0].name = 'Train Loss'
fig.data[1].name = 'Validation Loss'
fig.show()

# Get predictions for plotting
model.eval()
all_predictions = []
all_actual = []
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        all_predictions.extend(outputs.squeeze().tolist())
        all_actual.extend(targets.squeeze().tolist())

predictions = np.array(all_predictions)
actual = np.array(all_actual)


# Scatter plot of predicted vs actual prices
scatter_fig = px.scatter(x=actual, y=predictions, title='Actual vs. Predicted Prices (Test Set)')
scatter_fig.update_layout(
    xaxis_title='Actual Price',
    yaxis_title='Predicted Price'
)
scatter_fig.show()

# Distribution of predicted vs actual prices
results_df = pd.DataFrame({'Actual': actual, 'Predicted': predictions})
distribution_fig = px.histogram(results_df, x=['Actual', 'Predicted'], barmode='overlay', title='Distribution of Actual and Predicted Prices (Test Set)')
distribution_fig.show()

# Residual plot
residuals = actual - predictions
residuals_fig = px.scatter(x=predictions, y=residuals, title='Residual Plot (Test Set)')
residuals_fig.add_hline(y=0, line_dash="dash", line_color="red")
residuals_fig.update_layout(
    xaxis_title='Predicted Price',
    yaxis_title='Residual (Actual - Predicted)'
)
residuals_fig.show()

# Correlation Matrix (as before)
correlation_matrix = house.corr()
correlation_fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto", title='Correlation Matrix')
correlation_fig.show()

In [14]:
# prompt: i mean the conclusion with the results with r2 score

from sklearn.metrics import r2_score

# Calculate R-squared score on the test set
r2 = r2_score(actual, predictions)

print(f'R-squared (R2) Score on Test Set: {r2:.4f}')

print("\n--- Conclusion ---")
print("The neural network model was trained to predict house prices.")
print(f"After preprocessing and training for {epochs} epochs, the model achieved a Mean Squared Error (MSE) of {test_loss:.4f} on the test set.")
print(f"The R-squared (R2) score on the test set is {r2:.4f}.")
print("An R2 score closer to 1 indicates a better fit of the model to the data.")
if r2 > 0.75:
    print("The R2 score suggests that the model explains a significant portion of the variance in house prices.")
elif r2 > 0.5:
     print("The R2 score indicates that the model explains a moderate portion of the variance in house prices.")
else:
    print("The R2 score suggests that the model does not explain a large portion of the variance in house prices. Further model tuning or feature engineering might be needed.")
print("The training and validation loss curves show how the model's performance evolved during training.")
print("The scatter plot of actual vs. predicted prices visually represents the model's prediction accuracy.")
print("The residual plot helps assess the model's assumptions and identify potential areas for improvement.")


R-squared (R2) Score on Test Set: 0.8093

--- Conclusion ---
The neural network model was trained to predict house prices.
After preprocessing and training for 500 epochs, the model achieved a Mean Squared Error (MSE) of 319.2559 on the test set.
The R-squared (R2) score on the test set is 0.8093.
An R2 score closer to 1 indicates a better fit of the model to the data.
The R2 score suggests that the model explains a significant portion of the variance in house prices.
The training and validation loss curves show how the model's performance evolved during training.
The scatter plot of actual vs. predicted prices visually represents the model's prediction accuracy.
The residual plot helps assess the model's assumptions and identify potential areas for improvement.
